# Notebook 05 — Group Comparison
Compare two user groups (young vs elderly, patient vs control, pre vs post therapy).
Set `DATA_DIR_B` to a different folder of JSON files for the second group.

| Audience | What they get |
|---|---|
| **Researcher / HCI** | Statistical comparison with p-values and effect sizes |
| **Doctor / Clinician** | Which metrics separate the groups most strongly |
| **App maker** | Validate that the game produces different profiles for different populations |


In [1]:
DATA_DIR_A  = '.'     # Group A JSON folder
DATA_DIR_B  = '.'     # Group B JSON folder  ← change this
LABEL_A     = 'Group A'
LABEL_B     = 'Group B'
OUT_DIR     = 'outputs'
import os, json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── Make sure output folder exists BEFORE anything tries to write to it ──
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import (load_board_tries, load_bd_sessions, load_piano_sessions,
                        load_piano_movements, compare_groups, save_fig,
                        SHAPE_ORDER, HAND_COLORS, GROUP_COLORS)

df_a = load_board_tries(DATA_DIR_A)
df_b = load_board_tries(DATA_DIR_B)
sess_a, _ = load_piano_sessions(DATA_DIR_A)
sess_b, _ = load_piano_sessions(DATA_DIR_B)
print(f'{LABEL_A}: {len(df_a)} board tries, {len(sess_a)} piano sessions')
print(f'{LABEL_B}: {len(df_b)} board tries, {len(sess_b)} piano sessions')


KeyError: 'startedAt'

## Fig 05a — Shape success rate comparison

In [ ]:
shapes = [s for s in SHAPE_ORDER
          if s in df_a['shapeType'].values or s in df_b['shapeType'].values]

def shape_success(df):
    g = df.groupby('shapeType').agg(
        tries=('_id','count'), succ=('completed','sum')).reset_index()
    g['rate'] = g['succ'] / g['tries'] * 100
    return dict(zip(g['shapeType'], g['rate']))

sr_a = shape_success(df_a)
sr_b = shape_success(df_b)
x = np.arange(len(shapes))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, [sr_a.get(s, 0) for s in shapes], w,
       label=LABEL_A, color=GROUP_COLORS[0], alpha=0.85)
ax.bar(x + w/2, [sr_b.get(s, 0) for s in shapes], w,
       label=LABEL_B, color=GROUP_COLORS[1], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(shapes, rotation=20, ha='right')
ax.set_ylabel('Success rate (%)')
ax.set_ylim(0, 115)
ax.set_title('Shape success rate — Group comparison', fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig(fig, 'fig05a_shape_success_comparison.png', OUT_DIR)
plt.show()


## Fig 05b — Hand asymmetry comparison

In [ ]:
def hand_asym(df):
    g = df.groupby('hand').agg(
        succ=('completed','sum'), tries=('_id','count')).reset_index()
    g['rate'] = g['succ'] / g['tries'] * 100
    d = dict(zip(g['hand'], g['rate']))
    l = d.get('Left', np.nan)
    r = d.get('Right', np.nan)
    return l - r if not (np.isnan(l) or np.isnan(r)) else np.nan

asym_a = hand_asym(df_a)
asym_b = hand_asym(df_b)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar([LABEL_A, LABEL_B], [asym_a, asym_b],
              color=GROUP_COLORS, width=0.4, alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Left - Right success rate (pp)')
ax.set_title('Hand asymmetry score — Group comparison', fontweight='bold')
for bar, val in zip(bars, [asym_a, asym_b]):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                f'{val:.1f}pp', ha='center', fontsize=10)
plt.tight_layout()
save_fig(fig, 'fig05b_hand_asymmetry_comparison.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN / HCI SIGNAL ===')
print(f'{LABEL_A} asymmetry: {asym_a:.1f}pp')
print(f'{LABEL_B} asymmetry: {asym_b:.1f}pp')
print('Larger asymmetry = greater impairment. Useful baseline for clinical trials.')


## Fig 05c — Accuracy distribution violin comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
parts_a = ax.violinplot([df_a['accuracy'].dropna()], positions=[1], showmedians=True)
parts_b = ax.violinplot([df_b['accuracy'].dropna()], positions=[2], showmedians=True)
for pc in parts_a['bodies']:
    pc.set_facecolor(GROUP_COLORS[0]); pc.set_alpha(0.6)
for pc in parts_b['bodies']:
    pc.set_facecolor(GROUP_COLORS[1]); pc.set_alpha(0.6)
ax.set_xticks([1, 2])
ax.set_xticklabels([LABEL_A, LABEL_B])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy distribution — Group comparison', fontweight='bold')

res = compare_groups(df_a['accuracy'], df_b['accuracy'], LABEL_A, LABEL_B)
sig_label = 'SIGNIFICANT' if res['significant'] else 'not significant'
ax.text(0.5, -0.14,
        f"p={res['p_value']:.3f}  effect size={res['effect_size']:.2f}  {sig_label}",
        transform=ax.transAxes, ha='center', fontsize=9, color='gray')
plt.tight_layout()
save_fig(fig, 'fig05c_accuracy_violin.png', OUT_DIR)
plt.show()

print('\n=== RESEARCH SIGNAL ===')
print(f"Mean accuracy: {LABEL_A}={res['mean_a']:.1f}%  {LABEL_B}={res['mean_b']:.1f}%")
print(f"p={res['p_value']:.4f}  effect size (r)={res['effect_size']:.3f}")
print('Effect size interpretation: 0.1=small, 0.3=medium, 0.5=large')


## Fig 05d — Piano score and response time comparison

In [ ]:
valid_a = sess_a[sess_a['is_valid']]
valid_b = sess_b[sess_b['is_valid']]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title in [
    (axes[0], 'sessionScore', 'Piano session score'),
    (axes[1], 'avg_rt',       'Avg response time (s)'),
]:
    data = [valid_a[col].dropna(), valid_b[col].dropna()]
    bp = ax.boxplot(data, labels=[LABEL_A, LABEL_B], patch_artist=True,
                    medianprops=dict(color='black', lw=2))
    for patch, col_ in zip(bp['boxes'], GROUP_COLORS):
        patch.set_facecolor(col_); patch.set_alpha(0.6)
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
save_fig(fig, 'fig05d_piano_comparison.png', OUT_DIR)
plt.show()


## Fig 05e — Radar comparison

In [ ]:
def radar_vals(bd, piano):
    valid_p = piano[piano['is_valid']]
    avg_acc  = bd['accuracy'].mean() / 100 if len(bd) else 0
    gap      = abs(hand_asym(bd)) / 100 if len(bd) else 0
    symmetry = 1 - min(gap, 1)
    p_hit    = float(valid_p['hit_rate'].mean()) if len(valid_p) else 0
    p_rt     = 1 - min(float(valid_p['avg_rt'].mean()) / 3, 1) if len(valid_p) else 0
    p_score  = min(float(valid_p['sessionScore'].mean()) / 120, 1) if len(valid_p) else 0
    return [avg_acc, symmetry, p_hit, p_rt, p_score]

metrics = ['Avg accuracy', 'Hand symmetry', 'Piano hit rate', 'Response speed', 'Piano score']
angles  = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist() + [0]

va = radar_vals(df_a, sess_a); va = va + va[:1]
vb = radar_vals(df_b, sess_b); vb = vb + vb[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, va, color=GROUP_COLORS[0], lw=2, label=LABEL_A)
ax.fill(angles, va, color=GROUP_COLORS[0], alpha=0.15)
ax.plot(angles, vb, color=GROUP_COLORS[1], lw=2, label=LABEL_B)
ax.fill(angles, vb, color=GROUP_COLORS[1], alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=9)
ax.set_ylim(0, 1)
ax.set_title('Overall performance radar — Group comparison', fontweight='bold', pad=18)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
save_fig(fig, 'fig05e_radar_comparison.png', OUT_DIR)
plt.show()


## Stats summary table

In [ ]:
results = []
for metric, da, db in [
    ('Board accuracy (%)', df_a['accuracy'], df_b['accuracy']),
    ('Piano score',        valid_a['sessionScore'], valid_b['sessionScore']),
    ('Avg RT (s)',         valid_a['avg_rt'],       valid_b['avg_rt']),
]:
    res = compare_groups(da, db, LABEL_A, LABEL_B)
    results.append({
        'Metric': metric,
        'Mean_A' : round(res['mean_a'], 2),
        'Mean_B' : round(res['mean_b'], 2),
        'p_value': round(res['p_value'], 4),
        'effect_size': round(res['effect_size'], 3),
        'significant': res['significant'],
    })

stats_df = pd.DataFrame(results)
stats_df.to_csv(os.path.join(OUT_DIR, 'group_comparison_stats.csv'), index=False)
print('group_comparison_stats.csv saved')
print(stats_df.to_string(index=False))
